# Dataset loading and visual inspection

**The problem:** The CSVs use `;` as a delimiter instead of `,`.  
Pandas reads the whole row as one string and puts everything into a single column.

**The fix:** pass `sep=';'` to `pd.read_csv`.  
The cells below load both files correctly, clean the types, and display every view you need. need.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

# Paths
DATA_DIR = Path(os.getcwd()) / 'Data_0'

FILES = {
    'prices_d1': DATA_DIR / 'prices_round_0_day_-1.csv',
    'prices_d2': DATA_DIR / 'prices_round_0_day_-2.csv',
    'trades_d1': DATA_DIR / 'trades_round_0_day_-1.csv',
    'trades_d2': DATA_DIR / 'trades_round_0_day_-2.csv',
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Cannot find:\n' + '\n'.join(missing))

print('All four files found.')

All four files found.


#### the problem — what pandas gives you by default

In [3]:
# Read with the DEFAULT separator (comma) — shows the bug clearly
broken = pd.read_csv(FILES['prices_d1'])   # no sep argument

print(f'Shape when read with default comma separator: {broken.shape}')
print(f'Columns: {broken.columns.tolist()}')
print()
broken.head(4)

Shape when read with default comma separator: (20000, 1)
Columns: ['day;timestamp;product;bid_price_1;bid_volume_1;bid_price_2;bid_volume_2;bid_price_3;bid_volume_3;ask_price_1;ask_volume_1;ask_price_2;ask_volume_2;ask_price_3;ask_volume_3;mid_price;profit_and_loss']



,day;timestamp;product;bid_price_1;bid_volume_1;bid_price_2;bid_volume_2;bid_price_3;bid_volume_3;ask_price_1;ask_volume_1;ask_price_2;ask_volume_2;ask_price_3;ask_volume_3;mid_price;profit_and_loss
0,-1;0;TOMATOES;4999;5;4998;15;;;5013;5;5014;15;...
1,-1;0;EMERALDS;9992;14;9990;29;;;10008;14;10010...
2,-1;100;EMERALDS;9992;11;9990;22;;;10008;11;100...
3,-1;100;TOMATOES;5000;8;4998;21;;;5013;8;5014;2...


17 columns worth of data squeezed into a single column.  
Every row looks like one long string because pandas never found a comma to split on';'`

#### The fix: `sep=';'`

In [4]:
# Read with the CORRECT separator, semicolon
fixed = pd.read_csv(FILES['prices_d1'], sep=';')

print(f'Shape with sep=";": {fixed.shape}   ← 17 proper columns now')
print(f'Columns: {fixed.columns.tolist()}')
print()
fixed.head(4)

Shape with sep=";": (20000, 17)   ← 17 proper columns now
Columns: ['day', 'timestamp', 'product', 'bid_price_1', 'bid_volume_1', 'bid_price_2', 'bid_volume_2', 'bid_price_3', 'bid_volume_3', 'ask_price_1', 'ask_volume_1', 'ask_price_2', 'ask_volume_2', 'ask_price_3', 'ask_volume_3', 'mid_price', 'profit_and_loss']



,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-1,0,TOMATOES,4999,5,4998,15,NaN,NaN,5013,5,5014,15,NaN,NaN,5006.00,0.00
1,-1,0,EMERALDS,9992,14,9990,29,NaN,NaN,10008,14,10010,29,NaN,NaN,10000.00,0.00
2,-1,100,EMERALDS,9992,11,9990,22,NaN,NaN,10008,11,10010,22,NaN,NaN,10000.00,0.00
3,-1,100,TOMATOES,5000,8,4998,21,NaN,NaN,5013,8,5014,21,NaN,NaN,5006.50,0.00


#### Full loaders with type casting and derived columns

In [5]:
def load_prices(path: Path) -> pd.DataFrame:
    """
    Load a prices CSV.
    - Splits on semicolons (the actual delimiter)
    - Casts every numeric column (empty cells in level-2/3 columns become NaN)
    - Adds three derived columns: spread, total_bid_depth, total_ask_depth
    """
    df = pd.read_csv(path, sep=';')

    # Columns that should be numeric
    numeric_cols = [
        'timestamp',
        'bid_price_1', 'bid_volume_1',
        'bid_price_2', 'bid_volume_2',
        'bid_price_3', 'bid_volume_3',
        'ask_price_1', 'ask_volume_1',
        'ask_price_2', 'ask_volume_2',
        'ask_price_3', 'ask_volume_3',
        'mid_price', 'profit_and_loss',
    ]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    df['day'] = df['day'].astype(int)

    # Derived columns
    df['spread']          = df['ask_price_1'] - df['bid_price_1']
    df['total_bid_depth'] = df[['bid_volume_1', 'bid_volume_2', 'bid_volume_3']].sum(axis=1, min_count=1)
    df['total_ask_depth'] = df[['ask_volume_1', 'ask_volume_2', 'ask_volume_3']].sum(axis=1, min_count=1)

    return df

In [6]:
def load_trades(path: Path, day: int) -> pd.DataFrame:
    """
    Load a trades CSV.
    - Splits on semicolons
    - Adds the simulation day as a column (not present in the trades file)
    - Adds notional = price * quantity
    """
    df = pd.read_csv(path, sep=';')

    df['day']      = day
    df['price']    = pd.to_numeric(df['price'],    errors='coerce')
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df['notional'] = df['price'] * df['quantity']

    return df

In [7]:
# Load all four files
prices_d1 = load_prices(FILES['prices_d1'])
prices_d2 = load_prices(FILES['prices_d2'])
trades_d1 = load_trades(FILES['trades_d1'], day=-1)
trades_d2 = load_trades(FILES['trades_d2'], day=-2)

# Combined views across both days
prices_all = pd.concat([prices_d1, prices_d2], ignore_index=True)
trades_all = pd.concat([trades_d1, trades_d2], ignore_index=True)

print(f'prices_all  →  {prices_all.shape[0]:,} rows  ×  {prices_all.shape[1]} columns')
print(f'trades_all  →  {trades_all.shape[0]:,} rows  ×  {trades_all.shape[1]} columns')

prices_all  →  40,000 rows  ×  20 columns
trades_all  →  1,219 rows  ×  9 columns


#### Prices — what every column means

In [8]:
# Full column reference with dtypes
col_descriptions = {
    'day'            : 'Simulation day (-1 or -2)',
    'timestamp'      : 'Tick number (0 → 999,900 in steps of 100ms)',
    'product'        : 'EMERALDS or TOMATOES',
    'bid_price_1'    : 'Best (highest) bid price in the book',
    'bid_volume_1'   : 'Units available at best bid',
    'bid_price_2'    : 'Second-best bid price (next level down)',
    'bid_volume_2'   : 'Units at second bid level',
    'bid_price_3'    : 'Third bid level — empty on ~96% of ticks',
    'bid_volume_3'   : 'Units at third bid level',
    'ask_price_1'    : 'Best (lowest) ask price in the book',
    'ask_volume_1'   : 'Units available at best ask',
    'ask_price_2'    : 'Second-best ask price',
    'ask_volume_2'   : 'Units at second ask level',
    'ask_price_3'    : 'Third ask level — empty on ~96% of ticks',
    'ask_volume_3'   : 'Units at third ask level',
    'mid_price'      : '(bid_price_1 + ask_price_1) / 2',
    'profit_and_loss': 'Simulator bot cumulative PnL (always 0 in tutorial round)',
    'spread'         : 'DERIVED: ask_price_1 - bid_price_1',
    'total_bid_depth': 'DERIVED: sum of all bid volumes across levels 1-3',
    'total_ask_depth': 'DERIVED: sum of all ask volumes across levels 1-3',
}

ref = pd.DataFrame({
    'dtype'      : prices_all.dtypes.astype(str),
    'non_null'   : prices_all.notna().sum(),
    'null_pct'   : (prices_all.isna().mean() * 100).round(1).astype(str) + '%',
    'description': pd.Series(col_descriptions),
})

print('PRICES column reference')
display(ref)

PRICES column reference


,dtype,non_null,null_pct,description
day,int32,40000,0.0%,Simulation day (-1 or -2)
timestamp,int64,40000,0.0%,"Tick number (0 → 999,900 in steps of 100ms)"
product,object,40000,0.0%,EMERALDS or TOMATOES
bid_price_1,int64,40000,0.0%,Best (highest) bid price in the book
bid_volume_1,int64,40000,0.0%,Units available at best bid
bid_price_2,int64,40000,0.0%,Second-best bid price (next level down)
bid_volume_2,int64,40000,0.0%,Units at second bid level
bid_price_3,float64,1050,97.4%,Third bid level — empty on ~96% of ticks
bid_volume_3,float64,1050,97.4%,Units at third bid level
ask_price_1,int64,40000,0.0%,Best (lowest) ask price in the book


In [9]:
# First 10 rows of the clean prices dataframe — EMERALDS only
print('EMERALDS (Day -1) — first 10 rows')
display(prices_d1[prices_d1['product'] == 'EMERALDS'].head(10))

EMERALDS (Day -1) — first 10 rows


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread,total_bid_depth,total_ask_depth
1,-1,0,EMERALDS,9992,14,9990,29,NaN,NaN,10008,14,10010,29,NaN,NaN,10000.00,0.00,16,43.00,43.00
2,-1,100,EMERALDS,9992,11,9990,22,NaN,NaN,10008,11,10010,22,NaN,NaN,10000.00,0.00,16,33.00,33.00
4,-1,200,EMERALDS,9992,15,9990,20,NaN,NaN,10008,15,10010,20,NaN,NaN,10000.00,0.00,16,35.00,35.00
7,-1,300,EMERALDS,9992,11,9990,29,NaN,NaN,10008,11,10010,29,NaN,NaN,10000.00,0.00,16,40.00,40.00
9,-1,400,EMERALDS,9992,12,9990,25,NaN,NaN,10008,12,10010,25,NaN,NaN,10000.00,0.00,16,37.00,37.00
10,-1,500,EMERALDS,9992,12,9990,30,NaN,NaN,10008,12,10010,30,NaN,NaN,10000.00,0.00,16,42.00,42.00
12,-1,600,EMERALDS,9992,11,9990,25,NaN,NaN,10008,11,10010,25,NaN,NaN,10000.00,0.00,16,36.00,36.00
15,-1,700,EMERALDS,9992,12,9990,23,NaN,NaN,10008,12,10010,23,NaN,NaN,10000.00,0.00,16,35.00,35.00
17,-1,800,EMERALDS,9992,15,9990,29,NaN,NaN,10008,15,10010,29,NaN,NaN,10000.00,0.00,16,44.00,44.00
18,-1,900,EMERALDS,9992,15,9990,22,NaN,NaN,10008,15,10010,22,NaN,NaN,10000.00,0.00,16,37.00,37.00


In [10]:
# First 10 rows — TOMATOES only
print('TOMATOES (Day -1) — first 10 rows')
display(prices_d1[prices_d1['product'] == 'TOMATOES'].head(10))

TOMATOES (Day -1) — first 10 rows


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread,total_bid_depth,total_ask_depth
0,-1,0,TOMATOES,4999,5,4998,15,NaN,NaN,5013,5,5014,15,NaN,NaN,5006.00,0.00,14,20.00,20.00
3,-1,100,TOMATOES,5000,8,4998,21,NaN,NaN,5013,8,5014,21,NaN,NaN,5006.50,0.00,13,29.00,29.00
5,-1,200,TOMATOES,5000,10,4999,20,NaN,NaN,5013,10,5015,20,NaN,NaN,5006.50,0.00,13,30.00,30.00
6,-1,300,TOMATOES,5000,9,4999,21,NaN,NaN,5014,9,5015,21,NaN,NaN,5007.00,0.00,14,30.00,30.00
8,-1,400,TOMATOES,5000,5,4999,20,NaN,NaN,5014,5,5015,20,NaN,NaN,5007.00,0.00,14,25.00,25.00
11,-1,500,TOMATOES,5000,6,4999,25,NaN,NaN,5014,6,5015,25,NaN,NaN,5007.00,0.00,14,31.00,31.00
13,-1,600,TOMATOES,5005,5,5000,8,4999.00,25.00,5014,8,5015,25,NaN,NaN,5009.50,0.00,9,38.00,33.00
14,-1,700,TOMATOES,5000,10,4999,18,NaN,NaN,5014,10,5015,18,NaN,NaN,5007.00,0.00,14,28.00,28.00
16,-1,800,TOMATOES,5000,8,4998,15,NaN,NaN,5013,8,5014,15,NaN,NaN,5006.50,0.00,13,23.00,23.00
19,-1,900,TOMATOES,5000,6,4998,23,NaN,NaN,5013,6,5015,23,NaN,NaN,5006.50,0.00,13,29.00,29.00


In [11]:
# Spot-check a specific timestamp to read the book state at that moment
INSPECT_TS = 4200

print(f'Order book at timestamp {INSPECT_TS}')
snapshot = prices_d1[prices_d1['timestamp'] == INSPECT_TS]
display(snapshot)

print()
for _, row in snapshot.iterrows():
    product = row['product']
    print(f'{product}')
    print(f'  BID side:  L1 = {row["bid_price_1"]:.0f} × {row["bid_volume_1"]:.0f}', end='')
    
    if pd.notna(row['bid_price_2']):
        print(f'  |  L2 = {row["bid_price_2"]:.0f} × {row["bid_volume_2"]:.0f}', end='')
        
    if pd.notna(row['bid_price_3']):
        print(f'  |  L3 = {row["bid_price_3"]:.0f} × {row["bid_volume_3"]:.0f}', end='')
    print()
    print(f'  ASK side:  L1 = {row["ask_price_1"]:.0f} × {row["ask_volume_1"]:.0f}', end='')
    
    if pd.notna(row['ask_price_2']):
        print(f'  |  L2 = {row["ask_price_2"]:.0f} × {row["ask_volume_2"]:.0f}', end='')
        
    if pd.notna(row['ask_price_3']):
        print(f'  |  L3 = {row["ask_price_3"]:.0f} × {row["ask_volume_3"]:.0f}', end='')
    print()
    print(f'  Mid price: {row["mid_price"]:.1f}  |  Spread: {row["spread"]:.0f}  |  '
          f'Total bid depth: {row["total_bid_depth"]:.0f}  |  Total ask depth: {row["total_ask_depth"]:.0f}')
    print()

Order book at timestamp 4200


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread,total_bid_depth,total_ask_depth
84,-1,4200,EMERALDS,10000,9,9992,11,9990.00,24.00,10008,11,10010,24,NaN,NaN,10004.00,0.00,8,44.00,35.00
85,-1,4200,TOMATOES,4996,7,4994,25,NaN,NaN,5009,7,5010,25,NaN,NaN,5002.50,0.00,13,32.00,32.00



EMERALDS
  BID side:  L1 = 10000 × 9  |  L2 = 9992 × 11  |  L3 = 9990 × 24
  ASK side:  L1 = 10008 × 11  |  L2 = 10010 × 24
  Mid price: 10004.0  |  Spread: 8  |  Total bid depth: 44  |  Total ask depth: 35

TOMATOES
  BID side:  L1 = 4996 × 7  |  L2 = 4994 × 25
  ASK side:  L1 = 5009 × 7  |  L2 = 5010 × 25
  Mid price: 5002.5  |  Spread: 13  |  Total bid depth: 32  |  Total ask depth: 32



#### Trades — what every column means

In [12]:
trade_col_descriptions = {
    'timestamp': 'Tick when the trade executed',
    'buyer'    : 'Buyer ID — empty in tutorial round (named bots appear in later rounds)',
    'seller'   : 'Seller ID — empty in tutorial round',
    'symbol'   : 'EMERALDS or TOMATOES',
    'currency' : 'XIRECS',
    'price'    : 'Execution price (equals bid_price_1 or ask_price_1 at that tick)',
    'quantity' : 'Units traded',
    'day'      : 'DERIVED: simulation day (-1 or -2)',
    'notional' : 'DERIVED: price × quantity',
}

tref = pd.DataFrame({
    'dtype'      : trades_all.dtypes.astype(str),
    'non_null'   : trades_all.notna().sum(),
    'null_pct'   : (trades_all.isna().mean() * 100).round(1).astype(str) + '%',
    'description': pd.Series(trade_col_descriptions),
})

print('TRADES column reference')
display(tref)

TRADES column reference


,dtype,non_null,null_pct,description
timestamp,int64,1219,0.0%,Tick when the trade executed
buyer,float64,0,100.0%,Buyer ID — empty in tutorial round (named bots...
seller,float64,0,100.0%,Seller ID — empty in tutorial round
symbol,object,1219,0.0%,EMERALDS or TOMATOES
currency,object,1219,0.0%,XIRECS
price,float64,1219,0.0%,Execution price (equals bid_price_1 or ask_pri...
quantity,int64,1219,0.0%,Units traded
day,int64,1219,0.0%,DERIVED: simulation day (-1 or -2)
notional,float64,1219,0.0%,DERIVED: price × quantity


In [13]:
# All trades, Day -1
print('TRADES (Day -1) — all columns visible')
display(trades_d1)

TRADES (Day -1) — all columns visible


,timestamp,buyer,seller,symbol,currency,price,quantity,day,notional
0,3200,NaN,NaN,EMERALDS,XIRECS,9992.00,8,-1,79936.00
1,3400,NaN,NaN,TOMATOES,XIRECS,5009.00,2,-1,10018.00
2,5000,NaN,NaN,EMERALDS,XIRECS,9992.00,7,-1,69944.00
3,7000,NaN,NaN,TOMATOES,XIRECS,5010.00,4,-1,20040.00
4,9600,NaN,NaN,TOMATOES,XIRECS,4999.00,5,-1,24995.00
...,...,...,...,...,...,...,...,...,...
626,983800,NaN,NaN,EMERALDS,XIRECS,10008.00,3,-1,30024.00
627,985100,NaN,NaN,TOMATOES,XIRECS,4946.00,2,-1,9892.00
628,985600,NaN,NaN,EMERALDS,XIRECS,10008.00,6,-1,60048.00
629,993000,NaN,NaN,TOMATOES,XIRECS,4956.00,2,-1,9912.00


In [14]:
# Filter to a single product
print('EMERALDS trades (Day -1)')
display(trades_d1[trades_d1['symbol'] == 'EMERALDS'].reset_index(drop=True))

print()
print('TOMATOES trades (Day -1)')
display(trades_d1[trades_d1['symbol'] == 'TOMATOES'].reset_index(drop=True))

EMERALDS trades (Day -1)


,timestamp,buyer,seller,symbol,currency,price,quantity,day,notional
0,3200,NaN,NaN,EMERALDS,XIRECS,9992.00,8,-1,79936.00
1,5000,NaN,NaN,EMERALDS,XIRECS,9992.00,7,-1,69944.00
2,31400,NaN,NaN,EMERALDS,XIRECS,10008.00,3,-1,30024.00
3,39900,NaN,NaN,EMERALDS,XIRECS,9992.00,7,-1,69944.00
4,42400,NaN,NaN,EMERALDS,XIRECS,9992.00,5,-1,49960.00
...,...,...,...,...,...,...,...,...,...
203,973400,NaN,NaN,EMERALDS,XIRECS,10008.00,8,-1,80064.00
204,979200,NaN,NaN,EMERALDS,XIRECS,10008.00,6,-1,60048.00
205,982900,NaN,NaN,EMERALDS,XIRECS,10008.00,6,-1,60048.00
206,983800,NaN,NaN,EMERALDS,XIRECS,10008.00,3,-1,30024.00



TOMATOES trades (Day -1)


,timestamp,buyer,seller,symbol,currency,price,quantity,day,notional
0,3400,NaN,NaN,TOMATOES,XIRECS,5009.00,2,-1,10018.00
1,7000,NaN,NaN,TOMATOES,XIRECS,5010.00,4,-1,20040.00
2,9600,NaN,NaN,TOMATOES,XIRECS,4999.00,5,-1,24995.00
3,9900,NaN,NaN,TOMATOES,XIRECS,5000.00,4,-1,20000.00
4,16400,NaN,NaN,TOMATOES,XIRECS,4996.00,2,-1,9992.00
...,...,...,...,...,...,...,...,...,...
418,979000,NaN,NaN,TOMATOES,XIRECS,4965.00,5,-1,24825.00
419,982500,NaN,NaN,TOMATOES,XIRECS,4950.00,3,-1,14850.00
420,985100,NaN,NaN,TOMATOES,XIRECS,4946.00,2,-1,9892.00
421,993000,NaN,NaN,TOMATOES,XIRECS,4956.00,2,-1,9912.00


#### Summary statistics — clean and split by product

In [15]:
# Prices: key stats per product per day
key_price_cols = ['mid_price', 'spread', 'total_bid_depth', 'total_ask_depth']

for product in ['EMERALDS', 'TOMATOES']:
    print(f'\nPRICES — {product} (both days)')
    subset = prices_all[prices_all['product'] == product][key_price_cols]
    display(subset.describe().T.round(3))


PRICES — EMERALDS (both days)


,count,mean,std,min,25%,50%,75%,max
mid_price,20000.00,10000.00,0.72,9996.00,10000.00,10000.00,10000.00,10004.00
spread,20000.00,15.74,1.42,8.00,16.00,16.00,16.00,16.00
total_bid_depth,20000.00,37.67,3.71,30.00,35.00,38.00,40.00,55.00
total_ask_depth,20000.00,37.67,3.71,30.00,35.00,38.00,40.00,54.00



PRICES — TOMATOES (both days)


,count,mean,std,min,25%,50%,75%,max
mid_price,20000.00,4992.76,19.75,4946.50,4980.00,4995.50,5006.50,5036.00
spread,20000.00,13.02,1.75,5.00,13.00,13.00,14.00,14.00
total_bid_depth,20000.00,27.75,3.78,15.00,25.00,28.00,30.00,46.00
total_ask_depth,20000.00,27.75,3.75,20.00,25.00,28.00,30.00,47.00


In [16]:
# Trades: key stats per product
key_trade_cols = ['price', 'quantity', 'notional']

for product in ['EMERALDS', 'TOMATOES']:
    print(f'\nTRADES — {product} (both days)')
    subset = trades_all[trades_all['symbol'] == product][key_trade_cols]
    display(subset.describe().T.round(3))


TRADES — EMERALDS (both days)


,count,mean,std,min,25%,50%,75%,max
price,399.00,9999.80,7.90,9992.00,9992.00,10000.00,10008.00,10008.00
quantity,399.00,5.49,1.63,3.00,4.00,6.00,7.00,8.00
notional,399.00,54861.37,16257.46,29976.00,40032.00,59952.00,69944.00,80064.00



TRADES — TOMATOES (both days)


,count,mean,std,min,25%,50%,75%,max
price,820.00,4992.57,21.13,4943.00,4978.00,4994.00,5008.00,5040.00
quantity,820.00,3.48,1.12,2.00,2.00,3.00,4.00,6.00
notional,820.00,17372.84,5604.06,9888.00,10049.50,15072.00,20128.00,30186.00


In [17]:
# Spread value counts — the most important microstructure fingerprint
for product in ['EMERALDS', 'TOMATOES']:
    print(f'\n{product}: spread value counts (both days)')
    counts = (
        prices_all[prices_all['product'] == product]['spread']
        .value_counts()
        .sort_index()
        .rename('ticks')
        .to_frame()
    )
    counts['pct'] = (counts['ticks'] / counts['ticks'].sum() * 100).round(1).astype(str) + '%'
    display(counts)


EMERALDS: spread value counts (both days)


,ticks,pct
spread,,
8,654,3.3%
16,19346,96.7%



TOMATOES: spread value counts (both days)


,ticks,pct
spread,,
5,184,0.9%
6,253,1.3%
7,408,2.0%
8,466,2.3%
9,132,0.7%
13,9603,48.0%
14,8954,44.8%


In [18]:
# Trade count and notional by product and day
print('Trade activity summary')
display(
    trades_all
    .groupby(['symbol', 'day'])
    .agg(
        trade_count=('price', 'count'),
        mean_price=('price', 'mean'),
        mean_qty=('quantity', 'mean'),
        total_notional=('notional', 'sum'),
    )
    .round(2)
)

Trade activity summary


trade_count  mean_price  mean_qty  total_notional
symbol   day                                                   
EMERALDS -2           191     9999.66      5.45     10399728.00
         -1           208     9999.92      5.52     11489960.00
TOMATOES -2           397     5008.40      3.55      7067511.00
         -1           423     4977.72      3.41      7178215.00

#### Joining prices and trades at the same timestamp

This is what `TradingState` class defined by the exchange gives you every tick:  
the order book state **and** any trades that happened **at the same moment**.

In [19]:
# Merge prices and trades on timestamp + product/symbol
# Only rows where a trade occurred at that tick are kept
merged = pd.merge(
    left   = prices_d1,
    right  = trades_d1,
    left_on  = ['timestamp', 'product'],
    right_on = ['timestamp', 'symbol'],
    how      = 'inner',
    suffixes = ('_book', '_trade'),
)

# Which side was aggressive? trade price == ask → buyer crossed; == bid → seller crossed
merged['aggressor'] = np.where(
    np.abs(merged['price'] - merged['ask_price_1']) < 0.01, 'BUY',
    np.where(
        np.abs(merged['price'] - merged['bid_price_1']) < 0.01, 'SELL', 'UNKNOWN'
    )
)

view_cols = [
    'timestamp', 'product',
    'bid_price_1', 'mid_price', 'ask_price_1', 'spread',
    'price', 'quantity', 'aggressor'
]

print(f'Ticks where a trade occurred: {len(merged)}')
print()
print('Prices + trades joined (first 20 rows)')
display(merged[view_cols].head(20))

print()
print('Aggressor side breakdown per product')
display(
    merged.groupby(['product', 'aggressor']).size()
    .rename('count').reset_index()
)

Ticks where a trade occurred: 631

Prices + trades joined (first 20 rows)


,timestamp,product,bid_price_1,mid_price,ask_price_1,spread,price,quantity,aggressor
0,3200,EMERALDS,9992,10000.00,10008,16,9992.00,8,SELL
1,3400,TOMATOES,4996,5002.50,5009,13,5009.00,2,BUY
2,5000,EMERALDS,9992,9996.00,10000,8,9992.00,7,SELL
3,7000,TOMATOES,4997,5003.50,5010,13,5010.00,4,BUY
4,9600,TOMATOES,4999,5006.00,5013,14,4999.00,5,SELL
5,9900,TOMATOES,5000,5006.50,5013,13,5000.00,4,SELL
6,16400,TOMATOES,4996,5003.00,5010,14,4996.00,2,SELL
7,17900,TOMATOES,4995,5002.00,5009,14,5009.00,2,BUY
8,22200,TOMATOES,4991,4998.00,5005,14,5005.00,3,BUY
9,23500,TOMATOES,4989,4995.50,5002,13,4989.00,3,SELL



Aggressor side breakdown per product


,product,aggressor,count
0,EMERALDS,BUY,104
1,EMERALDS,SELL,104
2,TOMATOES,BUY,198
3,TOMATOES,SELL,225


#### Quick-access filter helpers

These one-liners let you slice the dataframes any way you need during exploration.

In [24]:
# All the ways to slice and inspect

# By product
em    = prices_all[prices_all['product'] == 'EMERALDS'].copy()
tom   = prices_all[prices_all['product'] == 'TOMATOES'].copy()

# By day
day1  = prices_all[prices_all['day'] == -1].copy()
day2  = prices_all[prices_all['day'] == -2].copy()

# By product and day
em_d1  = prices_all[(prices_all['product'] == 'EMERALDS') & (prices_all['day'] == -1)].copy()
em_d2  = prices_all[(prices_all['product'] == 'EMERALDS') & (prices_all['day'] == -2)].copy()
tom_d1 = prices_all[(prices_all['product'] == 'TOMATOES') & (prices_all['day'] == -1)].copy()
tom_d2 = prices_all[(prices_all['product'] == 'TOMATOES') & (prices_all['day'] == -2)].copy()

# Timestamps when spread is narrow (potential informed-order event)
em_narrow  = em_d1[em_d1['spread'] == 8]
tom_narrow = tom_d1[tom_d1['spread'] <= 8]

# Timestamps when level-3 book is visible
has_l3_bid = prices_all[prices_all['bid_price_3'].notna()]
has_l3_ask = prices_all[prices_all['ask_price_3'].notna()]

# Trade slices
em_trades  = trades_all[trades_all['symbol'] == 'EMERALDS']
tom_trades = trades_all[trades_all['symbol'] == 'TOMATOES']

print('Slice variables created:')
print(f'  em        {em.shape}    — EMERALDS prices, both days')
print(f'  tom       {tom.shape}    — TOMATOES prices, both days')
print(f'  em_d1     {em_d1.shape}  — EMERALDS Day -1 only')
print(f'  em_d2     {em_d2.shape}  — EMERALDS Day -2 only')
print(f'  tom_d1    {tom_d1.shape}  — TOMATOES Day -1 only')
print(f'  tom_d2    {tom_d2.shape}  — TOMATOES Day -2 only')
print(f'  em_narrow {em_narrow.shape}  — EMERALDS ticks where spread=8')
print(f'  tom_narrow {tom_narrow.shape} — TOMATOES ticks where spread<=8')
print(f'  has_l3_bid    {has_l3_ask.shape}  — Any product, ticks with 3-level Bid Order book')
print(f'  has_l3_ask    {has_l3_ask.shape}  — Any product, ticks with 3-level Ask Order book')
print(f'  em_trades  {em_trades.shape}  — EMERALDS trades only')
print(f'  tom_trades {tom_trades.shape}  — TOMATOES trades only')

Slice variables created:
  em        (20000, 20)    — EMERALDS prices, both days
  tom       (20000, 20)    — TOMATOES prices, both days
  em_d1     (10000, 20)  — EMERALDS Day -1 only
  em_d2     (10000, 20)  — EMERALDS Day -2 only
  tom_d1    (10000, 20)  — TOMATOES Day -1 only
  tom_d2    (10000, 20)  — TOMATOES Day -2 only
  em_narrow (321, 20)  — EMERALDS ticks where spread=8
  tom_narrow (670, 20) — TOMATOES ticks where spread<=8
  has_l3_bid    (1047, 20)  — Any product, ticks with 3-level Bid Order book
  has_l3_ask    (1047, 20)  — Any product, ticks with 3-level Ask Order book
  em_trades  (399, 9)  — EMERALDS trades only
  tom_trades (820, 9)  — TOMATOES trades only


In [22]:
# Inspect any slice interactively — change the variable and run
display(em_trades.head(10))

,timestamp,buyer,seller,symbol,currency,price,quantity,day,notional
0,3200,NaN,NaN,EMERALDS,XIRECS,9992.00,8,-1,79936.00
2,5000,NaN,NaN,EMERALDS,XIRECS,9992.00,7,-1,69944.00
12,31400,NaN,NaN,EMERALDS,XIRECS,10008.00,3,-1,30024.00
17,39900,NaN,NaN,EMERALDS,XIRECS,9992.00,7,-1,69944.00
18,42400,NaN,NaN,EMERALDS,XIRECS,9992.00,5,-1,49960.00
20,51300,NaN,NaN,EMERALDS,XIRECS,9992.00,7,-1,69944.00
25,55300,NaN,NaN,EMERALDS,XIRECS,9992.00,6,-1,59952.00
26,55700,NaN,NaN,EMERALDS,XIRECS,9992.00,4,-1,39968.00
28,62700,NaN,NaN,EMERALDS,XIRECS,9992.00,8,-1,79936.00
31,66700,NaN,NaN,EMERALDS,XIRECS,10008.00,7,-1,70056.00


In [25]:
display(has_l3_ask.head(10))

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread,total_bid_depth,total_ask_depth
57,-1,2800,EMERALDS,9992,11,9990,25,NaN,NaN,10000,10,10008,11,10010.00,25.00,9996.00,0.00,8,36.00,46.00
86,-1,4300,TOMATOES,4995,6,4994,25,NaN,NaN,5002,6,5008,6,5010.00,25.00,4998.50,0.00,7,31.00,37.00
100,-1,5000,EMERALDS,9992,13,9990,28,NaN,NaN,10000,9,10008,13,10010.00,28.00,9996.00,0.00,8,41.00,50.00
212,-1,10600,TOMATOES,5002,6,5000,25,NaN,NaN,5009,4,5015,6,5016.00,25.00,5005.50,0.00,7,31.00,35.00
236,-1,11800,TOMATOES,4999,8,4997,25,NaN,NaN,5006,5,5012,8,5013.00,25.00,5002.50,0.00,7,33.00,38.00
267,-1,13300,EMERALDS,9992,13,9990,24,NaN,NaN,10000,10,10008,13,10010.00,24.00,9996.00,0.00,8,37.00,47.00
278,-1,13900,TOMATOES,4997,9,4996,15,NaN,NaN,5004,6,5011,9,5012.00,15.00,5000.50,0.00,7,24.00,30.00
293,-1,14600,TOMATOES,4997,9,4995,22,NaN,NaN,5004,2,5010,9,5011.00,22.00,5000.50,0.00,7,31.00,33.00
318,-1,15900,TOMATOES,4998,5,4996,16,NaN,NaN,5005,6,5011,5,5012.00,16.00,5001.50,0.00,7,21.00,27.00
323,-1,16100,TOMATOES,4997,10,4996,24,NaN,NaN,5005,2,5011,10,5012.00,24.00,5001.00,0.00,8,34.00,36.00


In [26]:
display(has_l3_bid.head(10))

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,spread,total_bid_depth,total_ask_depth
13,-1,600,TOMATOES,5005,5,5000,8,4999.00,25.00,5014,8,5015,25,NaN,NaN,5009.50,0.00,9,38.00,33.00
22,-1,1100,TOMATOES,5007,8,5000,5,4999.00,17.00,5013,5,5015,17,NaN,NaN,5010.00,0.00,6,30.00,22.00
30,-1,1500,TOMATOES,5008,8,5000,7,4999.00,21.00,5014,7,5015,21,NaN,NaN,5011.00,0.00,6,36.00,28.00
84,-1,4200,EMERALDS,10000,9,9992,11,9990.00,24.00,10008,11,10010,24,NaN,NaN,10004.00,0.00,8,44.00,35.00
114,-1,5700,TOMATOES,5003,8,4995,9,4994.00,18.00,5009,9,5010,18,NaN,NaN,5006.00,0.00,6,35.00,27.00
176,-1,8800,EMERALDS,10000,10,9992,14,9990.00,25.00,10008,14,10010,25,NaN,NaN,10004.00,0.00,8,49.00,39.00
274,-1,13700,TOMATOES,5006,5,4998,6,4997.00,23.00,5012,6,5013,23,NaN,NaN,5009.00,0.00,6,34.00,29.00
298,-1,14900,EMERALDS,10000,5,9992,12,9990.00,26.00,10008,12,10010,26,NaN,NaN,10004.00,0.00,8,43.00,38.00
299,-1,14900,TOMATOES,5001,4,4996,10,4995.00,22.00,5009,10,5011,22,NaN,NaN,5005.00,0.00,8,36.00,32.00
349,-1,17400,TOMATOES,5001,6,4996,5,4995.00,19.00,5010,5,5011,19,NaN,NaN,5005.50,0.00,9,30.00,24.00
